#### get text content from data
If you use this dataset, please cite the following paper:
Urchs et al. (2025): taz2024full: Analysing German Newspapers for Gender Bias and Discrimination across Decades, Findings of ACL 2025.

format:
- meta data:

    - "published_on": publishing date in the format YYYY-MM-DDThh:mm:ss+01:00

    - "contains_actors": boolean indicating whether person entities were detected

    - "crawled_on": crawling date

    - "language": always "de"

    - "type": always "article"

    - "author": article author

    - "keywords": topic keywords

    - "token_count": number of tokens


- text:

    - "title": article title

    - "teaser": short summary

    - "text": main body of the article

In [2]:
import pandas as pd
import ast
import json
from pprint import pprint
import os
import re
import string

#import spacy
#nlp = spacy.load('de_core_news_lg') # oder de_core_news_sm

def flatten(xss):
    return [x for xs in xss for x in xs]

In [3]:
c_directory = r"corpus" 

# Iterate over files in directory
dict_year = {}
for name in os.listdir(c_directory):
    dir = os.path.join(c_directory, name)
    for n in os.listdir(dir):
            with open(os.path.join(dir, n)) as f:
                data = json.load(f)
                dict_year[name] = data
print(dict_year['2013'].keys())

dict_keys(['501045', '501051', '501053', '501055', '501056', '501058', '501062', '501064', '501065', '501079', '501081', '501096', '501098', '501102', '501105', '501108', '501111', '501114', '501118', '501121', '501123', '501127', '501130', '501131', '501133', '501134', '501136', '501143', '501148', '501150', '501152', '501154', '501156', '501158', '501161', '501166', '501170', '501172', '501174', '501182', '501185', '501188', '501193', '501199', '501210', '501213', '501215', '501221', '501223', '501225', '501227', '501229', '501233', '501235', '501239', '501240', '501242', '501244', '501245', '501268', '501270', '501271', '501272', '501273', '501274', '501277', '501279', '501281', '501282', '501283', '501284', '501285', '501286', '501287', '501288', '501290', '501291', '501292', '501293', '501294', '501295', '501296', '501297', '501299', '501300', '501301', '501302', '501303', '501304', '501305', '501306', '501307', '501309', '501310', '501311', '501312', '501313', '501314', '501315',

In [ ]:
# separate sentencesy with spacy
def separate_sentences(sents):
    doc = nlp(sents)
    res = []
    for sent in doc.sents:
     res.append(sent.text.lower())
     # entfernen von vorkommen von "h hb hh ki", tags?
    return res

In [4]:
# keep text content and id only
def save_txt_to_file():
    text_dir = "corpus_texts/"
    # Create the directory
    try:
        os.mkdir(text_dir)
        print(f"Directory '{text_dir}' created successfully.")
    except FileExistsError:
        print(f"Directory '{text_dir}' already exists.")

    #fill dir
    for key in dict_year.keys():
        text_dict = {}
        for k in dict_year[key].keys():
            text = dict_year[key][k]['text']['text']
            text_dict[k] = text
            with open(f"{text_dir}_{k}_full.txt", "w") as f:
                    f.write(json.dumps(str(text))) # separate_sentences(str(text))

save_txt_to_file()

Directory 'corpus_texts/' created successfully.


#### get sentences with occurences of AI related words
Notes: 
- Englisch and German for certain terms?
- Specific product names
- ML/AI system terminology
- Nouns only
- Abreviations, too



steps:
- build AI vocabulary
- filter by sentence containing at least one AI vocab
- multiply sentences if they contain multiple terms 

In [5]:
# AI-terms from ANTHROSCORE paper
ai_terms = list(ast.literal_eval(open("config_files/ger_anthroscore_KI_terms.txt").read()).values())

# get AI-related words, sorted according to 'Häufigkeitsklassen'
# https://corpora.uni-leipzig.de/de/res?corpusId=deu_news_2024&word=KI
ki_terms = ['KI','Künstliche Intelligenz', 'ChatGPT', 'AI', 'Copilot', 'Gemini', 'Chatbot','LLM', 'Chat-GPT',
            'Sprachmodell', 'KI-Modell', 'KI-System', 'KI-Tool', 'Artificial Intelligence', 'KI-Anwendung'] + ai_terms
ki_terms = [x.lower() for x in ki_terms]
regex_ki = "".join(["\\b"+x.replace(" ","\s")+"\\b|" for x in ki_terms])[:-1]
print(regex_ki)

\bki\b|\bkünstliche\sintelligenz\b|\bchatgpt\b|\bai\b|\bcopilot\b|\bgemini\b|\bchatbot\b|\bllm\b|\bchat-gpt\b|\bsprachmodell\b|\bki-modell\b|\bki-system\b|\bki-tool\b|\bartificial\sintelligence\b|\bki-anwendung\b|\bpalm\b|\blms\b|\bllama\b|\btransformer\b|\bsprachmodelle\b|\bsprachmodell\b|\bgpt\b|\bplms\b|\bvortrainierte\ssprachmodelle\b|\bgpt-2\b|\bxlnet\b|\bgroße\ssprachmodelle\b|\bllms\b|\bgpt-3\b|\bstiftungsmodell\b|\bgpt-neo\b|\bgpt-j\b|\bchatgpt\b|\bgpt-4\b


<>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
/var/folders/0h/lr036jk55nq7b5pmf0kg0qq40000gn/T/ipykernel_1921/4085195894.py:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  regex_ki = "".join(["\\b"+x.replace(" ","\s")+"\\b|" for x in ki_terms])[:-1]


#### Sätze säubern, filtern
- Satzzeichen
- (Html, etc)
- POS (use spacy bc TIGER corpus trained)
- Lexikon für Verb- / Adjektivtypen

In [6]:
# preprocessing auxiliary functions
''' 
stopwords = nlp.Defaults.stop_words
with open(f"config_files/spacy_stopwords.txt", "w") as f:
        f.write(str(stopwords))
'''

# this one is not needed if working on Sentence_spacy
def remove_punctuation(corpus):
    cleaned_corpus = corpus.translate(str.maketrans('', '', string.punctuation))
    return(cleaned_corpus)

# remove stop words from text
def remove_stop_words_from_text(cleaned_corpus):
    out = [word for word in cleaned_corpus.lower().split() if not word in stopwords]
    out_text = ' '.join(out)
    return out_text  # , out


def remove_stop_words(token_list):
    try:
        out = [token for token in token_list if not token.is_stop]
        return out
    except:
        return None

def determine_negation(doc):
    negation_tokens = [tok for tok in doc if tok.dep_ == 'neg']
    negation_head_tokens = [token.head for token in negation_tokens]
    for token in negation_head_tokens:
        return [child for child in token.children]


##### get sentences with ki_terms

In [11]:
# filter ki_terms
c_txt_dir = r"corpus_texts" 
ki_sentences = {}
for dir in os.listdir(c_txt_dir):
    if dir.endswith('_full.txt'):
        with open(os.path.join(c_txt_dir,dir)) as f:
            sent_l = f.read() #json.load(f)
            # could also account for lemmata, etc
        sent_l = [remove_punctuation(sent) for sent in sent_l]
        res_tmp = []
        for sent in sent_l:
            if re.search(regex_ki, sent):
            #print("res",sent, re.search(regex_ki, sent).string)
                res_tmp.append(sent)
            # add to dict
            if len(res_tmp) > 0:
                ki_sentences [dir] =  res_tmp

# save to file
filtered_sentences_json = json.dumps(ki_sentences)

pprint(len(filtered_sentences_json))
pprint(filtered_sentences_json)

2
'{}'


In [8]:
#save dict to file
with open("generated_data/KI_texts_full.json", "w") as f:
        f.write(str(filtered_sentences_json))


### Daten veranschaulichen:
- Häufigkeit der KI Terme
- Häufigste Wörter
- length of sentences / number of clauses
- number / type of verbs
- number / type of adjektives

In [ ]:
plots_dir = "plots/"